# IEEE PES | Recovering a quadratic Hessian from regional means: 2D and 3D

Companion to `quadratic_hessian.ipynb` (2D, point evaluations). Here
$f(x)=c+b^\top x+\tfrac12 x^\top Qx$ on $[0,1]^d$ with a known symmetric $Q$, so the true Hessian is $Q$.
Every estimate is built only from **regional means** of $f$ over boxes, the quantity a regression tree stores,
through the tree gradient estimate of Wycoff (2024, eq. 3), $\gamma = 2(\mu_r-\mu_l)/(u-l)$.

Four cases are run for each dimension:

* **(a)** aligned, equally spaced cells with *exact* regional means: deterministic approximation error only;
* **(b)** the same cells with *sample* means from $n$ uniform points: sampling error only;
* **(c)** the geometry of a fitted CART tree with exact means, using the uneven-split (effective-centre) extension:
  tests whether the tree's partition identifies $Q$ at all;
* **(d)** the same CART tree with its own leaf means: what a tree actually delivers.

Code lives in `tree_hessian/` (this folder); full tables in `experiments/results/`; findings in `report_3d_hessian.md`.

In [1]:
import sys, pathlib, numpy as np
sys.path.insert(0, str(pathlib.Path.cwd()))   # run from weeks/2026-W45/
from sklearn.tree import DecisionTreeRegressor
from tree_hessian import Quadratic, extract_nodes, find_diag_configs, find_mixed_configs
from tree_hessian.quadratic import box_mean_sample
from tree_hessian.stencils import diag_four_cell, mixed_four_cell, cells_diag_four, cells_mixed_four
from tree_hessian.uneven import hessian_from_tree, symmetrize
from tree_hessian.synthetic import with_exact_means
np.set_printoptions(precision=4, suppress=True)


def compare(H, Q, label):
    """Print the known Q, the estimate H, their entrywise difference, and the max/Frobenius error."""
    print(label)
    print("Known Hessian Q:\n", Q)
    print("Estimated Hessian:\n", H)
    print("Entrywise error:\n", H - Q)
    print("Maximum absolute error: %.3e    Frobenius error: %.3e\n"
          % (np.nanmax(np.abs(H - Q)), np.linalg.norm(np.nan_to_num(H - Q))))


def aligned_estimate(mean_fn, lo, hi):
    """All Hessian entries from the aligned equal-cell stencils on the box [lo, hi],
    with cell means supplied by mean_fn(lo, hi) (exact or sampled)."""
    # diagonal: four equal slabs along x_i; mixed: 2x2 equal cells in the (x_i, x_j) plane
    d = len(lo); H = np.full((d, d), np.nan)
    for i in range(d):
        mu = [mean_fn(*cell) for cell in cells_diag_four(lo, hi, i)]
        H[i, i] = diag_four_cell(*mu, lo[i], hi[i])
    for i in range(d):
        for j in range(i + 1, d):
            mu = [mean_fn(*cell) for cell in cells_mixed_four(lo, hi, i, j)]
            H[i, j] = H[j, i] = mixed_four_cell(*mu, lo[i], hi[i], lo[j], hi[j])
    return H


def run_all(Q, b, c, depth, N, n_samples=(10**4, 10**6)):
    """Run cases (a)-(d) for the quadratic (Q, b, c) on [0,1]^d: aligned cells with exact
    and sampled means, then a CART tree of the given depth fit to N points."""
    d = Q.shape[0]; f = Quadratic(Q, b, c)
    lo, hi = np.zeros(d), np.ones(d)
    pairs = [(i, j) for i in range(d) for j in range(i + 1, d)]

    H = aligned_estimate(f.box_mean_exact, lo, hi)
    compare(H, Q, "(a) aligned cells, exact regional means on [0,1]^%d" % d)
    np.testing.assert_allclose(H, Q, rtol=0, atol=1e-10)

    for n in n_samples:
        X, y = f.sample_uniform(lo, hi, n, np.random.default_rng(0))
        compare(aligned_estimate(lambda a, b_: box_mean_sample(X, y, a, b_)[0], lo, hi), Q,
                "(b) aligned cells, sample means, n = %d uniform points, noiseless y" % n)

    X, y = f.sample_uniform(lo, hi, N, np.random.default_rng(0))
    reg = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, y)
    nodes = extract_nodes(reg, lo, hi)
    n_aligned = sum(len(find_diag_configs(nodes, i)) for i in range(d)) + \
                sum(len(find_mixed_configs(nodes, i, j)) for i, j in pairs)
    print("CART depth %d, N = %d: subtrees meeting the aligned/equal-cell assumptions: %d\n" % (depth, N, n_aligned))

    Q_raw, info = hessian_from_tree(with_exact_means(nodes, f.box_mean_exact), d)
    compare(symmetrize(Q_raw)[0], Q, "(c) CART geometry + exact regional means (effective-centre extension)")

    Q_raw, info = hessian_from_tree(nodes, d)
    print("rank of the pair system per row:", {s: int(v["rank"]) for s, v in info.items()}, "(need %d)" % d)
    compare(symmetrize(Q_raw)[0], Q, "(d) CART depth %d, N = %d, its own leaf means" % (depth, N))

## 1. Two dimensions

The quadratic of `quadratic_hessian.ipynb`: $Q=\begin{pmatrix}4&1.5\\1.5&3\end{pmatrix}$, $b=(-1,2)$, $c=0.7$.
That notebook reconstructed $Q$ from point evaluations with step $h=10^{-3}$ (max error $6\times10^{-10}$);
here the same $Q$ is reconstructed from regional means.

In [2]:
Q2 = np.array([[4.0, 1.5],
               [1.5, 3.0]])
run_all(Q2, b=np.array([-1.0, 2.0]), c=0.7, depth=6, N=20_000)

(a) aligned cells, exact regional means on [0,1]^2
Known Hessian Q:
 [[4.  1.5]
 [1.5 3. ]]
Estimated Hessian:
 [[4.  1.5]
 [1.5 3. ]]
Entrywise error:
 [[ 0. -0.]
 [-0.  0.]]
Maximum absolute error: 3.553e-15    Frobenius error: 7.105e-15

(b) aligned cells, sample means, n = 10000 uniform points, noiseless y
Known Hessian Q:
 [[4.  1.5]
 [1.5 3. ]]
Estimated Hessian:
 [[3.6804 1.3668]
 [1.3668 2.8153]]
Entrywise error:
 [[-0.3196 -0.1332]
 [-0.1332 -0.1847]]
Maximum absolute error: 3.196e-01    Frobenius error: 4.144e-01

(b) aligned cells, sample means, n = 1000000 uniform points, noiseless y
Known Hessian Q:
 [[4.  1.5]
 [1.5 3. ]]
Estimated Hessian:
 [[4.0632 1.5011]
 [1.5011 2.9841]]
Entrywise error:
 [[ 0.0632  0.0011]
 [ 0.0011 -0.0159]]
Maximum absolute error: 6.315e-02    Frobenius error: 6.514e-02



CART depth 6, N = 20000: subtrees meeting the aligned/equal-cell assumptions: 0

(c) CART geometry + exact regional means (effective-centre extension)
Known Hessian Q:
 [[4.  1.5]
 [1.5 3. ]]
Estimated Hessian:
 [[4.  1.5]
 [1.5 3. ]]
Entrywise error:
 [[-0.  0.]
 [ 0. -0.]]
Maximum absolute error: 2.220e-15    Frobenius error: 3.109e-15

rank of the pair system per row: {0: 2, 1: 2} (need 2)
(d) CART depth 6, N = 20000, its own leaf means
Known Hessian Q:
 [[4.  1.5]
 [1.5 3. ]]
Estimated Hessian:
 [[4.1008 1.482 ]
 [1.482  2.9503]]
Entrywise error:
 [[ 0.1008 -0.018 ]
 [-0.018  -0.0497]]
Maximum absolute error: 1.008e-01    Frobenius error: 1.152e-01



## 2. Three dimensions

$Q=\begin{pmatrix}4&1.5&0.5\\1.5&3&-1\\0.5&-1&2\end{pmatrix}$, $b=(-1,2,0.5)$, $c=0.7$.

In [3]:
Q3 = np.array([[4.0, 1.5, 0.5],
               [1.5, 3.0, -1.0],
               [0.5, -1.0, 2.0]])
run_all(Q3, b=np.array([-1.0, 2.0, 0.5]), c=0.7, depth=7, N=20_000)

(a) aligned cells, exact regional means on [0,1]^3
Known Hessian Q:
 [[ 4.   1.5  0.5]
 [ 1.5  3.  -1. ]
 [ 0.5 -1.   2. ]]
Estimated Hessian:
 [[ 4.   1.5  0.5]
 [ 1.5  3.  -1. ]
 [ 0.5 -1.   2. ]]
Entrywise error:
 [[ 0. -0.  0.]
 [-0. -0. -0.]
 [ 0. -0.  0.]]
Maximum absolute error: 1.066e-14    Frobenius error: 1.404e-14

(b) aligned cells, sample means, n = 10000 uniform points, noiseless y
Known Hessian Q:
 [[ 4.   1.5  0.5]
 [ 1.5  3.  -1. ]
 [ 0.5 -1.   2. ]]
Estimated Hessian:
 [[ 4.2021  1.4188  0.5771]
 [ 1.4188  3.0206 -1.1469]
 [ 0.5771 -1.1469  1.5926]]
Entrywise error:
 [[ 0.2021 -0.0812  0.0771]
 [-0.0812  0.0206 -0.1469]
 [ 0.0771 -0.1469 -0.4074]]
Maximum absolute error: 4.074e-01    Frobenius error: 5.249e-01



(b) aligned cells, sample means, n = 1000000 uniform points, noiseless y
Known Hessian Q:
 [[ 4.   1.5  0.5]
 [ 1.5  3.  -1. ]
 [ 0.5 -1.   2. ]]
Estimated Hessian:
 [[ 4.0496  1.5286  0.507 ]
 [ 1.5286  3.0003 -0.9721]
 [ 0.507  -0.9721  2.0414]]
Entrywise error:
 [[0.0496 0.0286 0.007 ]
 [0.0286 0.0003 0.0279]
 [0.007  0.0279 0.0414]]
Maximum absolute error: 4.961e-02    Frobenius error: 8.636e-02

CART depth 7, N = 20000: subtrees meeting the aligned/equal-cell assumptions: 0

(c) CART geometry + exact regional means (effective-centre extension)
Known Hessian Q:
 [[ 4.   1.5  0.5]
 [ 1.5  3.  -1. ]
 [ 0.5 -1.   2. ]]
Estimated Hessian:
 [[ 4.   1.5  0.5]
 [ 1.5  3.  -1. ]
 [ 0.5 -1.   2. ]]
Entrywise error:
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Maximum absolute error: 5.329e-15    Frobenius error: 6.822e-15

rank of the pair system per row: {0: 3, 1: 3, 2: 3} (need 3)
(d) CART depth 7, N = 20000, its own leaf means
Known Hessian Q:
 [[ 4.   1.5  0.5]
 [ 1.5  3.  -1. ]
 [ 0.5 -1.   2

## 3. Reading the errors

* (a) is $10^{-14}$ in both dimensions: the aligned construction is exact for a quadratic.
* (b) and (d) are sampling error in cell means; it falls with $n$ at fixed geometry and is larger in 3D, where each cell holds fewer points.
* CART never produces the aligned configuration (zero subtrees), so (c) and (d) need the effective-centre extension.
* (c) is $10^{-15}$: the partition CART produces is sufficient once every variable is split by enough nodes (rank $d$ per row).

For a quadratic target the bias of the tree estimate is zero; what remains is variance.